## 1. Setup

We will first install a number of libraries and import what we will need.





In [1]:
#%%capture
!pip install -q -U transformers
!pip install -q -U datasets loralib sentencepiece
!pip install -q bitsandbytes accelerate
!pip install -q langchain
!pip install einops
!pip install faiss-gpu
!pip install langchain_community
!pip install --upgrade --quiet chromadb bs4 qdrant-client
!pip install langchainhub
!pip install -U langchain-huggingface
!pip install -U langchain-cohere # Upgrade langchain-cohere
!pip install --upgrade --quiet  wikipedia
!pip install --upgrade --quiet  arxiv
!pip install --upgrade --quiet  pymupdf

!pip install xmltodict
!pip uninstall -y cohere
!pip install cohere==4.52 # Install a compatible version
!pip install langchain langchain-community langchain-core
!pip install -U langchain langchain-community langchain-core huggingface_hub

In [ ]:
!pip install -U qdrant-client langchain-qdrant

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
pip install --upgrade langchain-community qdrant-client

In [2]:
import torch
import os
import bs4
import json
import numpy as np
import time


from pprint import pprint

import locale

from transformers import AutoTokenizer , AutoModelForCausalLM
from transformers import pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
# from langchain.llms import HuggingFacePipeline
from langchain_community.chat_models import ChatCohere
#from langchain_cohere import ChatCohere
# from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
# from langchain.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
# from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores import Qdrant
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.utils.math import cosine_similarity

from langchain_community.document_loaders import ArxivLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.document_loaders import OnlinePDFLoader
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import PubMedLoader


from google.colab import userdata

In [3]:
import langchain
print(langchain.__version__)

1.2.9


In [4]:
locale.getpreferredencoding = lambda: "UTF-8"

In [5]:
#%%capture
!pip install -U sentence_transformers

Add your keys from the secret store (do **NOT** print them out or leave them exposed as plaintext in your notebook!):

In [129]:
'77HAocwKFSWSTVTm4ghDUQp5kZ1kOVG9Q7CSo1HT'

'77HAocwKFSWSTVTm4ghDUQp5kZ1kOVG9Q7CSo1HT'

In [7]:
COHERE_API_KEY = userdata.get('COHERE_API_KEY')

In [8]:
#%%capture
base_embeddings = HuggingFaceEmbeddings(model_name="multi-qa-mpnet-base-dot-v1")

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 18.9 MB/s eta 0:00:00


In [ ]:
cohere_chat_model = ChatCohere(cohere_api_key=COHERE_API_KEY)

## **Import context : PDF & Wiki**

In [110]:
import asyncio
from langchain_community.document_loaders import PyPDFLoader

urls = [
    "https://static1.squarespace.com/static/5ea39a89c76f373be702d49a/t/5ea4b427ecc98b1707b46c4e/1587852328673/CBEC+2013.pdf",
    "https://scielo.org.za/pdf/sajip/v36n1/v36n1a11.pdf",
    "https://www.ccl.org/wp-content/uploads/2015/05/coach-coachee-characteristics-research-paper-center-for-creative-leadership.pdf",
    "https://files.eric.ed.gov/fulltext/ED478147.pdf",
    "https://knowledgecommons.lakeheadu.ca/server/api/core/bitstreams/3c13df26-086c-4d5b-b3e3-0d8c4e41ce0b/content",
    "https://www.siop.org/wp-content/uploads/2024/07/SHRM-SIOP_Executive_coaching.pdf",
    "https://infonomics-society.org/wp-content/uploads/ijibs/published-papers/volume-2-2016/Insight-as-a-Turning-Point-for-Learning-in-Executive-Coaching.pdf"
]

async def load_all_pdfs(url_list):
    docs = []
    for url in url_list:
        try:
            loader = PyPDFLoader(url)
            data = loader.load()
            docs.extend(data)
            print(f"load successfully: {url[:50]}...")
        except Exception as e:
            print(f"error {url}: {e}")
    return docs

all_docs = await load_all_pdfs(urls)

print(f" {len(all_docs)} downloaded")



load successfully: https://static1.squarespace.com/static/5ea39a89c76...
load successfully: https://scielo.org.za/pdf/sajip/v36n1/v36n1a11.pdf...
load successfully: https://www.ccl.org/wp-content/uploads/2015/05/coa...
load successfully: https://files.eric.ed.gov/fulltext/ED478147.pdf...
load successfully: https://knowledgecommons.lakeheadu.ca/server/api/c...
load successfully: https://www.siop.org/wp-content/uploads/2024/07/SH...
load successfully: https://infonomics-society.org/wp-content/uploads/...
 450 downloaded


In [130]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=200,
    add_start_index=True
)

In [131]:
splits = text_splitter.split_documents(all_docs)

In [132]:
#index doc chunks
for idx, text in enumerate(splits):
    splits[idx].metadata['split_id'] = idx

print('Number of splits/chunks: ', len(splits))

Number of splits/chunks:  1170


In [ ]:
print(splits[0])

In [134]:
qdrant_vectorstore = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=base_embeddings,
    location=":memory:",
    collection_name="private_coach_rag",
    force_recreate=True
)

In [135]:
retriever = qdrant_vectorstore.as_retriever(search_kwargs={"k": 3})

## test load:

In [140]:
query = "How can we be a good prviate coach?"
found_docs = qdrant_vectorstore.similarity_search_with_score(query)

In [141]:
print(found_docs[0][0].page_content)
print(found_docs[0][1])

focus on developing that relationship. These skills include being a good listener, having a need to 
understand the process and the coachee. There is an emphasis on having respect for each other and 
coming from an empathic place.
3. Balance of Challenge & Support. Coaches need the ability to both confront and strengthen a 
coachee through helpful and caring suggestions, as well as pushing the coachee beyond where he or 
she feels comfortable. Coaches must also provide both positive and constructive feedback.
4. Credibility. Coaches need to show they have a background in coaching, through experience, 
education, degrees, certifications, and business acumen. Credibility also entails how genuine the 
coach is in the work that he or she does.
5. Adaptable. A coach should be adaptive, flexible, and be able to adjust to best meet the needs 
of the coaching relationship. A coach needs to be willing to make changes to style and structure to 
best accomplish the goals of the coaching.
0.600787

## Load more from Wiki

In [146]:
wiki_docs = WikipediaLoader(query="Executive Coaching", load_max_docs=4).load()
for idx, text in enumerate(wiki_docs):
    wiki_docs[idx].metadata['doc_num'] = global_doc_number
    wiki_docs[idx].metadata['doc_source'] = "Wikipedia"

    global_doc_number += 1

print('Number of documents: ', len(wiki_docs))

#index docs
wiki_splits = text_splitter.split_documents(wiki_docs)
for idx, text in enumerate(wiki_splits):
    wiki_splits[idx].metadata['split_id'] = idx

print('Number of splits/chunks: ', len(wiki_splits))




Number of documents:  4
Number of splits/chunks:  23


In [147]:
#%%capture

qdrant_vectorstore.add_documents(documents=wiki_splits)

Same with a couple of other queries:

In [148]:
wiki_docs = WikipediaLoader(query="Cognitive behavioral therapy", load_max_docs=4).load()
for idx, text in enumerate(wiki_docs):
    wiki_docs[idx].metadata['doc_num'] = global_doc_number
    wiki_docs[idx].metadata['doc_source'] = "Wikipedia"

    global_doc_number += 1

print('Number of documents: ', len(wiki_docs))

#index docs
wiki_splits = text_splitter.split_documents(wiki_docs)
for idx, text in enumerate(wiki_splits):
    wiki_splits[idx].metadata['split_id'] = idx

print('Number of splits/chunks: ', len(wiki_splits))




Number of documents:  4
Number of splits/chunks:  25


In [149]:
#%%capture

qdrant_vectorstore.add_documents(documents=wiki_splits)

And yet another related Wikipedia article.

In [150]:
wiki_docs = WikipediaLoader(query="Growth mindset", load_max_docs=4).load()
for idx, text in enumerate(wiki_docs):
    wiki_docs[idx].metadata['doc_num'] = global_doc_number
    wiki_docs[idx].metadata['doc_source'] = "Wikipedia"

    global_doc_number += 1

print('Number of documents: ', len(wiki_docs))

#index docs
wiki_splits = text_splitter.split_documents(wiki_docs)
for idx, text in enumerate(wiki_splits):
    wiki_splits[idx].metadata['split_id'] = idx

print('Number of splits/chunks: ', len(wiki_splits))

Number of documents:  4
Number of splits/chunks:  27


In [151]:
#%%capture

qdrant_vectorstore.add_documents(documents=wiki_splits)

And finally another related Wikipedia article.

In [152]:
wiki_docs = WikipediaLoader(query="Metacognition", load_max_docs=4).load()
for idx, text in enumerate(wiki_docs):
    wiki_docs[idx].metadata['doc_num'] = global_doc_number
    wiki_docs[idx].metadata['doc_source'] = "Wikipedia"

    global_doc_number += 1

print('Number of documents: ', len(wiki_docs))

#index docs
wiki_splits = text_splitter.split_documents(wiki_docs)
for idx, text in enumerate(wiki_splits):
    wiki_splits[idx].metadata['split_id'] = idx

print('Number of splits/chunks: ', len(wiki_splits))

Number of documents:  4
Number of splits/chunks:  21


In [153]:
#%%capture

qdrant_vectorstore.add_documents(documents=wiki_splits)

In [78]:
from operator import itemgetter

In [164]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [170]:
ls /content/drive/MyDrive/pdf

'A Guide to Rational Living.pdf'
'Cognitive Behavior Therapy - Basics and Beyond.pdf'
'Difficult Conversations How to Discuss What Matters Most.pdf'
'DVBT Skills Training Handouts and Worksheets.pdf'
'Getting to Yes.pdf'
'Mind Over Mood.pdf'
'Motivational Interviewing - Helping People Change and Grow.pdf'
'Nonviolent Communication - A Language of Life.pdf'
'Relevant Context.docx'


In [174]:
import os
from langchain_community.document_loaders import PyPDFLoader

# Path to your Google Drive folder
folder_path = "/content/drive/MyDrive/pdf"
drive_docs = []

# 1. Get all PDF files, filtering out hidden "._" files and non-PDFs
pdf_files = [f for f in os.listdir(folder_path)
             if f.endswith('.pdf') and not f.startswith('._')]

print(f"🔍 Found {len(pdf_files)} potential PDF files. Starting ingestion...")

# 2. Iterate and load with error handling
for file_name in pdf_files:
    file_path = os.path.join(folder_path, file_name)
    try:
        # We use the standard PyPDFLoader for individual files
        loader = PyPDFLoader(file_path)
        data = loader.load()
        drive_docs.extend(data)
        print(f"✅ Successfully loaded: {file_name} ({len(data)} pages)")
    except Exception as e:
        # This catches 'PdfStreamError' or 'invalid pdf header' without stopping the script
        print(f"❌ Skipping corrupted or invalid file: {file_name}")
        print(f"   Reason: {e}")

# 3. Add Metadata for RAG tracing
for doc in drive_docs:
    doc.metadata['source_type'] = "Google_Drive_Coach_Data"

print(f"\n🚀 Ingestion Complete! Total pages loaded: {len(drive_docs)}")

🔍 Found 8 potential PDF files. Starting ingestion...
✅ Successfully loaded: Motivational Interviewing - Helping People Change and Grow.pdf (446 pages)
✅ Successfully loaded: Mind Over Mood.pdf (355 pages)
✅ Successfully loaded: DVBT Skills Training Handouts and Worksheets.pdf (414 pages)
✅ Successfully loaded: Cognitive Behavior Therapy - Basics and Beyond.pdf (194 pages)


✅ Successfully loaded: A Guide to Rational Living.pdf (246 pages)
❌ Skipping corrupted or invalid file: Difficult Conversations How to Discuss What Matters Most.pdf
   Reason: Stream has ended unexpectedly
✅ Successfully loaded: Getting to Yes.pdf (7 pages)
✅ Successfully loaded: Nonviolent Communication - A Language of Life.pdf (272 pages)

🚀 Ingestion Complete! Total pages loaded: 1934


In [175]:
drive_splits = text_splitter.split_documents(drive_docs)

for idx, text in enumerate(web_splits):
    drive_splits[idx].metadata['split_id'] = idx

print('Number of splits: ', len(web_splits))

Number of splits:  276


In [176]:
#%%capture

qdrant_vectorstore.add_documents(documents=web_splits)

In [177]:
retriever = qdrant_vectorstore.as_retriever()

### DATA PREP

In [181]:
pip install jq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 14.7 MB/s eta 0:00:00


In [190]:
## load json file
import json
from langchain_core.documents import Document

# Path to your file
file_path = '/content/drive/MyDrive/Bed002_meeting_transcript.json'

def load_custom_transcript(path):
    with open(path, 'r') as f:
        data = json.load(f)

    docs = []
    # Loop through each string in the transcript list
    for entry in data['transcript']:
        # Split by ':' to separate ID, Speaker, Timestamp, and Text
        # Based on your image: "MeetingID : Speaker : Timestamp : Text"
        parts = entry.split(':', 3)

        if len(parts) == 4:
            meeting_id, speaker, timestamp, text = parts

            # Create a LangChain Document
            doc = Document(
                page_content=text.strip(),
                metadata={
                    "meeting_id": meeting_id,
                    "speaker": speaker,
                    "timestamp": timestamp,
                    "source_type": "Transcript"
                }
            )
            docs.append(doc)

    return docs

# Execute the loader
transcript_docs = load_custom_transcript(file_path)

print(f"✅ Successfully loaded {len(transcript_docs)} lines.")
print(f"Sample Metadata: {transcript_docs[0].metadata}")
print(f"Sample Text: {transcript_docs[0].page_content}")

✅ Successfully loaded 1045 lines.
Sample Metadata: {'meeting_id': 'Bed002', 'speaker': 'B', 'timestamp': '25.42', 'source_type': 'Transcript'}
Sample Text: So , which is my bar ? Mah ! Number


In [193]:
def get_strategic_segments(docs, window_size=150):
    total = len(docs)
    # 1. Opening: First 150 lines (Setting the stage)
    opening = docs[:window_size]

    # 2. Middle: 150 lines from the center (The core discussion)
    mid_start = total // 2 - (window_size // 2)
    middle = docs[mid_start : mid_start + window_size]

    # 3. Closing: Last 150 lines (The results/agreements)
    closing = docs[-window_size:]

    return opening, middle, closing

# Get the segments
opening_docs, middle_docs, closing_docs = get_strategic_segments(transcript_docs)
print(opening_docs)

[Document(metadata={'meeting_id': 'Bed002', 'speaker': 'B', 'timestamp': '25.42', 'source_type': 'Transcript'}, page_content='So , which is my bar ? Mah ! Number'), Document(metadata={'meeting_id': 'Bed002', 'speaker': 'C', 'timestamp': '54.17', 'source_type': 'Transcript'}, page_content='Other way . We m We may wind up with'), Document(metadata={'meeting_id': 'Bed002', 'speaker': 'C', 'timestamp': '56.28', 'source_type': 'Transcript'}, page_content='ver We we may need versions of all this garbage'), Document(metadata={'meeting_id': 'Bed002', 'speaker': 'B', 'timestamp': '58.21', 'source_type': 'Transcript'}, page_content='one . For our stuff . Yeah . OK .'), Document(metadata={'meeting_id': 'Bed002', 'speaker': 'B', 'timestamp': '58.31', 'source_type': 'Transcript'}, page_content='This is Transcript three six three one three six five'), Document(metadata={'meeting_id': 'Bed002', 'speaker': 'B', 'timestamp': '58.41', 'source_type': 'Transcript'}, page_content='O . three one five four t

In [195]:

contextualize_q_system_prompt = """
You are a Senior Cognitive Coaching Analyst. Your job is to transform a raw conversation transcript into actionable intelligence.

### TASK 1: DIAGNOSTIC SCORING (CRI & CEI)
Evaluate the dialogue dynamics based on the following metrics:
- CRI (Conflict Resolution): Sentiment slope, Accountability (taking ownership), and Solution Progress (Phase 1-4).
- CEI (Effectiveness): Conversational Balance (Word share %), Q&A Completion, and Convergence.

### TASK 2: MULTI-QUERY GENERATION (Cognitive Focus)
Generate exactly 3 search queries for the PDF Knowledge Base to help the coach:
1. **Clarification Query**: To resolve specific technical or conceptual ambiguities that cause mental friction.
2. **Behavioral/Emotional Query**: To find psychological frameworks or "Cognitive Biases" (e.g., social loafing, fear of failure, or dominance) that explain the current CRI/CEI bottlenecks.
3. **Intervention Query**: To find specific coaching prompts or "Reframing techniques" to shift the team's emotional state from defensive/passive to proactive/collaborative.

### SPECIAL INSTRUCTION: TREND ANALYSIS
You are provided with three chronological segments: [OPENING], [MIDDLE], and [CLOSING].
1. Compare the CRI/CEI scores across these segments.
2. Did the Solution Progress move forward or backward?
3. In your Bottleneck analysis, identify if the issue is 'Improving', 'Persistent', or 'Worsening'.

### OUTPUT FORMAT
Your output must follow this structure EXACTLY:
---
[CRI/CEI ANALYSIS]
- Metrics: (Provide score details)
- Bottleneck: (Describe the main coaching challenge detected)
---
[SEARCH QUERIES]
1. (Query 1)
2. (Query 2)
3. (Query 3)
"""


In [196]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Helper function to format segments into a readable string for the LLM
def format_segment(docs, label):
    text = "\n".join([
        f"[{d.metadata['timestamp']}] Speaker {d.metadata['speaker']}: {d.page_content}"
        for d in docs
    ])
    return f"### {label} SEGMENT ###\n{text}\n"

# 2. Combine the segments into one cohesive input for the Trend Analysis
combined_transcript = (
    format_segment(opening_docs, "OPENING") + "\n" +
    format_segment(middle_docs, "MIDDLE") + "\n" +
    format_segment(closing_docs, "CLOSING")
)

# 3. Set up the diagnostic chain using your prompt
diagnostic_prompt_template = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    ("human", "Please analyze the following conversation segments for trends:\n\n{chat_history}")
])

diagnostic_chain = diagnostic_prompt_template | cohere_chat_model | StrOutputParser()

# 4. Execute the analysis
print("--- Starting Trend Analysis (Opening + Middle + Closing) ---\n")
full_diagnostic_report = diagnostic_chain.invoke({"chat_history": combined_transcript})

print(full_diagnostic_report)

--- Starting Trend Analysis (Opening + Middle + Closing) ---

---
[CRI/CEI ANALYSIS]
- **Metrics**:  
  - **CRI (Conflict Resolution)**:  
    - Sentiment Slope: Neutral to slightly positive across segments, indicating no major conflicts.  
    - Accountability: Low. Ownership of tasks is unclear, with vague assignments (e.g., "you guys who got the email").  
    - Solution Progress: Phase 2 (Exploration) in [OPENING], regressing to Phase 1 (Problem Definition) in [MIDDLE], and returning to Phase 2 in [CLOSING].  
  - **CEI (Effectiveness)**:  
    - Conversational Balance: Imbalanced (Speaker C dominates with 65% word share).  
    - Q&A Completion: Incomplete. Questions like "So are the people going to be identified by name?" are left unresolved.  
    - Convergence: Low. Team diverges into technical details without aligning on core objectives.  

- **Bottleneck**: Persistent lack of clarity on task ownership and deliverables, compounded by imbalanced participation and unresolved que

In [200]:
# Debug Cell: Let's see the raw output lines
print("--- Raw Diagnostic Report Lines ---")
report_lines = full_diagnostic_report.strip().split("\n")
for i, line in enumerate(report_lines):
    print(f"Line {i}: |{line}|")

--- Raw Diagnostic Report Lines ---
Line 0: |---|
Line 1: |[CRI/CEI ANALYSIS]|
Line 2: |- **Metrics**:  |
Line 3: |  - **CRI (Conflict Resolution)**:  |
Line 4: |    - Sentiment Slope: Neutral to slightly positive across segments, indicating no major conflicts.  |
Line 5: |    - Accountability: Low. Ownership of tasks is unclear, with vague assignments (e.g., "you guys who got the email").  |
Line 6: |    - Solution Progress: Phase 2 (Exploration) in [OPENING], regressing to Phase 1 (Problem Definition) in [MIDDLE], and returning to Phase 2 in [CLOSING].  |
Line 7: |  - **CEI (Effectiveness)**:  |
Line 8: |    - Conversational Balance: Imbalanced (Speaker C dominates with 65% word share).  |
Line 9: |    - Q&A Completion: Incomplete. Questions like "So are the people going to be identified by name?" are left unresolved.  |
Line 10: |    - Convergence: Low. Team diverges into technical details without aligning on core objectives.  |
Line 11: ||
Line 12: |- **Bottleneck**: Persistent lac

In [202]:
# 1. Initialize the list and split report into lines
extracted_queries = []
lines = full_diagnostic_report.split('\n')

# 2. Iterate through each line to find queries
for i in range(len(lines)):
    current_line = lines[i].strip()

    # Identify lines starting with the numbers 1, 2, or 3
    if current_line.startswith(('1.', '2.', '3.')):
        # Check if the next line exists to extract the actual query text
        if i + 1 < len(lines):
            next_line = lines[i+1].strip()

            # Extract content between double quotes
            if '"' in next_line:
                # Using split to isolate the text inside the quotes
                query = next_line.split('"')[1]
                extracted_queries.append(query)

# 3. Verify the extraction results
print("--- Final Extracted Search Queries ---")
for idx, q in enumerate(extracted_queries, 1):
    print(f"Query {idx}: {q}")

--- Final Extracted Search Queries ---
Query 1: How to operationalize 'intentions' (e.g., Vista mode, Tango mode) in a belief-net model for navigation systems?
Query 2: Psychological frameworks for addressing social loafing in team meetings with dominant speakers and passive participants.
Query 3: Reframing techniques to shift defensive/passive teams to proactive collaboration in technical discussions.


In [203]:
# 1. Retrieve scientific context for each of the 3 queries
all_retrieved_docs = []

print("--- 🔍 Accessing Knowledge Base ---")
for query in extracted_queries:
    print(f"Retrieving insights for: {query[:50]}...")
    # This uses your existing retriever to fetch the top relevant segments
    docs = retriever.invoke(query)
    all_retrieved_docs.extend(docs)

# 2. Combine the retrieved text into a single context block
pdf_context = "\n---\n".join([doc.page_content for doc in all_retrieved_docs])

--- 🔍 Accessing Knowledge Base ---
Retrieving insights for: How to operationalize 'intentions' (e.g., Vista mo...
Retrieving insights for: Psychological frameworks for addressing social loa...
Retrieving insights for: Reframing techniques to shift defensive/passive te...


In [204]:
# 3. Define the Final Synthesis Prompt
# This prompt tells the AI to act as a coach, combining the CRI/CEI diagnosis with the PDF facts.
final_synthesis_template = """
[INST]
You are a World-Class Cognitive Coach. You have analyzed a transcript and retrieved specific scientific insights from your knowledge base.

### 1. DIAGNOSTIC SUMMARY (Current State):
{diagnostic_report}

### 2. SCIENTIFIC CONTEXT (Retrieved Knowledge):
{pdf_context}

### YOUR TASK:
Using the Scientific Context provided, provide a high-impact coaching plan:
1. **Root Cause Analysis**: Explain the psychological or technical reason for the 'Bottleneck' identified in the report.
2. **Key Coaching Prompts**: Provide 3 powerful questions the coach should ask Speaker C (the dominant speaker) or Speaker B (the passive speaker) to shift the dynamic.
3. **Actionable Intervention**: Suggest one specific "Reframing Exercise" or "Meeting Protocol" the team can use immediately.

### RESPONSE FORMAT:
- **Root Cause Analysis**: [Brief explanation]
- **Strategic Questions**: [List of 3 questions]
- **Recommended Intervention**: [One clear exercise]
[/INST]
"""

In [205]:
# 4. Run the Final Chain
final_coach_prompt = ChatPromptTemplate.from_template(final_synthesis_template)
final_synthesis_chain = final_coach_prompt | cohere_chat_model | StrOutputParser()

print("\n--- 🎓 Generating Final Coaching Strategy ---")
final_response = final_synthesis_chain.invoke({
    "diagnostic_report": full_diagnostic_report,
    "pdf_context": pdf_context
})

print("\n" + "="*50)
print(final_response)
print("="*50)


--- 🎓 Generating Final Coaching Strategy ---

**Root Cause Analysis**:  
The bottleneck—persistent lack of clarity on task ownership and deliverables, compounded by imbalanced participation—stems from a combination of **social loafing** (passive participants avoiding accountability) and **goal ambiguity** (unclear operationalization of intentions/modes in the belief-net model). Speaker C’s dominance likely triggers **psychological reactance** in others, reducing their engagement, while the team’s regression from Phase 2 (Exploration) to Phase 1 (Problem Definition) reflects a **self-regulatory cycle disruption** (Sharp, 1997). The quadratic reciprocity between thoughts, emotions, and behavior (Beck et al., 1979) is disrupted, as vague assignments and unresolved questions create cognitive dissonance, hindering proactive collaboration.  

**Strategic Questions**:  
1. **To Speaker C**: *"How might your insights be amplified if others shared their perspectives more actively, and what ste

**## Build Chat Loop**

In [207]:
!pip install -q langchain-community

In [208]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [213]:
# 1. Initialize message history
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

history = ChatMessageHistory()

# 2. Dynamic Speaker Detection
all_speakers = list(set([d.metadata['speaker'] for d in transcript_docs]))
speaker_list_str = ", ".join(all_speakers)

# 3. Define the Dynamic Chat Loop with User Identification
def chat_with_coach_assistant():
    print(f"🎓 Cognitive Coach Assistant is Online.")
    print(f"Detected Speakers: {speaker_list_str}")
    print("-" * 50)

    # NEW: Identify the User's Role
    user_role = input(f"Before we begin, which speaker are you? ({speaker_list_str}): ").strip()

    # Update initial context with user identity
    initial_context = f"""
    You are a World-Class Cognitive Coaching Expert Assistant.
    You are coaching a human user who is one of the participants in the meeting.

    [USER IDENTITY]
    The user is: {user_role}

    [PARTICIPANTS DETECTED]
    Speakers in this session: {speaker_list_str}

    [CASE DATA & DIAGNOSIS]
    {final_response}

    [SCIENTIFIC CONTEXT FROM PDFS]
    {pdf_context[:2000]}

    [YOUR MISSION]
    - If the user is a dominant speaker, help them with "stepping back" and "active listening."
    - If the user is a passive speaker, help them with "assertiveness" and "reclaiming ownership."
    - Use the scientific context to explain the 'why' behind their emotions or behaviors.
    - Always speak directly to {user_role} as the coach.
    """

    print(f"\n✅ Understood. I am now acting as the personal coach for {user_role}.")
    print("(Type 'exit', 'quit', or 'stop' to end)")
    print("-" * 50)

    while True:
        user_input = input(f"{user_role} (You): ")
        if user_input.lower() in ['exit', 'quit', 'stop']:
            print("Chat session ended.")
            break

        messages = [SystemMessage(content=initial_context)]
        messages.extend(history.messages)
        messages.append(HumanMessage(content=user_input))

        try:
            response = cohere_chat_model.invoke(messages)
            ai_message = response.content if hasattr(response, 'content') else str(response)

            history.add_user_message(user_input)
            history.add_ai_message(ai_message)

            print(f"\nAssistant: {ai_message}\n" + "-" * 30)

        except Exception as e:
            print(f"❌ Execution Error: {e}")

# 4. Start the interaction
chat_with_coach_assistant()

🎓 Cognitive Coach Assistant is Online.
Detected Speakers: B, C
--------------------------------------------------
Before we begin, which speaker are you? (B, C): B

✅ Understood. I am now acting as the personal coach for B.
(Type 'exit', 'quit', or 'stop' to end)
--------------------------------------------------
B (You): what should I do, I feel bad

Assistant: It’s completely normal to feel this way, B, especially when there’s ambiguity around task ownership and participation. Let’s break this down using the cognitive-behavioral framework we discussed. Here’s how we can address this:

1. **Thoughts**:  
   What specific thoughts are contributing to you feeling bad? Is it uncertainty about what’s expected of you, or perhaps frustration with the lack of clarity? Identifying these thoughts is the first step to reframing them. For example, instead of thinking, *"I don’t know what to do,"* you could shift to, *"I can clarify what’s expected and take ownership of a specific task."*

2. **F